# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-1-'
comment = "acute_all-simple_d_E"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 1248)] if x not in run_list]
print(len(out))
print(' '.join((out)))

0



In [4]:
num_cpu = 150
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
# mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
#                                                                                                         for file_name in file_list)), 
#                        columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()
full_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names])

# # Save datasets
full_df.to_pickle(os.path.join(d, "raw", "stacked_full_data"+runs+"runs"+'-'+comment)+'.pkl')

In [5]:
with pd.option_context('display.max_columns', None):
    display(full_df)

,S_0,I_0,b_I,d_S,d_I,d_IE,K_I,d_H,K_H,K_S,N_0,max_Na,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_cycle,psi_myc_I,psi_myc_HI,psi_myc_HE,L0_Na,psi_NE_I,psi_NE_HI,psi_NE_HE,L0_NE,psi_EM_I,psi_EM_HI,psi_EM_HE,L0_EM,psi_Edie_I,psi_Edie_HI,psi_Edie_HE,L0_Edie,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_eM,T_pEcyteM,T_pE_end,frac_cM,int_pHE,int_pHI,E_end
0,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.333333,-0.333333,-0.333333,-2.5,-1.333333,-0.333333,-0.333333,2.0,-0.0,-0.0,-0.0,-81.0,1.333333,0.333333,0.333333,-0.5,2.047469e-07,0.00,0.02,1.010000e+03,3.731648e+03,60.0,3.17,30.00,0.0,0.0,0.00,0.152284,5.151520e+02,2.530733e+02,0.0
1,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.333333,-0.333333,-0.333333,-2.5,-1.333333,-0.333333,-0.333333,2.0,-0.0,-0.0,-0.0,-81.0,1.333333,0.333333,0.333333,0.0,2.389371e-07,0.00,0.02,1.010000e+03,2.188793e+03,55.0,4.99,4.97,0.0,0.0,0.00,0.238342,4.444838e+02,2.533359e+02,0.0
2,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.333333,-0.333333,-0.333333,-2.5,-1.333333,-0.333333,-0.333333,2.0,-0.0,-0.0,-0.0,-81.0,1.333333,0.333333,0.333333,0.5,2.472712e-07,0.00,0.02,1.010000e+03,1.845665e+03,37.0,3.73,30.00,0.0,0.0,0.00,0.221080,2.744543e+02,2.533748e+02,0.0
3,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.333333,-0.333333,-0.333333,-2.5,-1.333333,-0.333333,-0.333333,2.0,-0.0,-0.0,-0.0,-81.0,1.333333,0.333333,0.333333,1.0,2.612954e-07,0.00,0.02,1.010000e+03,1.295193e+03,36.0,5.54,30.00,0.0,0.0,0.00,0.184143,2.130348e+02,2.531697e+02,0.0
4,10000000.0,1000.0,5.000000e-08,0.01,0.5,12.0,10000.0,1.0,100000.0,10000000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.333333,-0.333333,-0.333333,-2.5,-1.333333,-0.333333,-0.333333,2.0,-0.0,-0.0,-0.0,-81.0,1.333333,0.333333,0.333333,1.5,2.599908e-07,0.00,0.02,1.010000e+03,1.345106e+03,39.0,2.76,30.00,0.0,0.0,0.00,0.163683,2.179607e+02,2.533299e+02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390971524,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,10000000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.333333,-0.666667,0.666667,3.0,-0.333333,-0.666667,0.666667,3.0,-0.0,-0.0,0.0,-81.0,0.333333,0.666667,-0.666667,-2.5,8.547318e-12,3.67,6.57,1.395392e+06,8.605611e+06,2558188.0,6.78,0.83,0.0,0.0,30.00,0.154613,3.951877e+06,1.932327e+05,4020.0
390971525,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,10000000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.333333,-0.666667,0.666667,3.0,-0.333333,-0.666667,0.666667,3.0,-0.0,-0.0,0.0,-81.0,0.333333,0.666667,-0.666667,-2.0,4.929352e-09,3.96,7.23,2.349171e+06,7.651824e+06,2198641.0,7.26,1.08,0.0,0.0,29.43,0.188119,3.747832e+06,3.340748e+05,69.0
390971526,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,10000000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.333333,-0.666667,0.666667,3.0,-0.333333,-0.666667,0.666667,3.0,-0.0,-0.0,0.0,-81.0,0.333333,0.666667,-0.666667,-1.5,5.684562e-07,4.16,7.93,3.113213e+06,6.887490e+06,1832331.0,5.91,1.18,0.0,0.0,21.99,0.141439,3.414430e+06,4.533711e+05,0.0
390971527,10000000.0,1000.0,2.500000e-07,0.01,0.5,12.0,10000000.0,1.0,100000.0,10000000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-0.333333,-0.666667,0.666667,3.0,-0.333333,-0.666667,0.666667,3.0,-0.0,-0.0,0.0,-81.0,0.333333,0.666667,-0.666667,-1.0,1.767654e-05,4.30,9.30,3.890226e+06,6.104967e+06,1393524.0,5.82,1.15,0.0,0.0,17.50,0.159601,3.283246e+06,5.795630e+05,0.0


In [6]:
# Create additional variables
virs = np.unique(full_df[['I_0','d_I','K_I','b_I','K_H','N_0']].values, axis = 0)

full_df['antigenicity_over_harm'] = antigenicity_over_harm(full_df)
full_df['stim_pI'] = np.log(1 + (full_df['p_load']/full_df['K_I']))
full_df['stim_pHI'] = np.log(1 + (full_df['int_pHI']/full_df['K_H']))
full_df['stim_pHE'] = np.log(1 + (full_df['int_pHE']/full_df['K_H']))
#full_df['scaled_min_pS'] = full_df['min_pS']/full_df['S_0']

# identify Biologically evidenced networks
keep_vars = ['harm_pI', 'harm_pS', 'max_pE',
             'T_pE_start', 'T_pE_max', 'T_pE_end',
             'stim_pI', 'stim_pHI', 'stim_pHE',
             'E_end', 'antigenicity_over_harm']

In [7]:
# save data sets
full_infection_scenarios = []
mean_of_infection_scenarios = []
std_of_infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_I, b_I, K_H, N_0) in enumerate(tqdm(virs)):
    data = full_df.loc[(full_df["d_I"] == d_I)*(full_df["K_I"] == K_I)*(full_df["b_I"] == b_I)*(full_df["K_H"] == K_H)*(full_df["N_0"] == N_0)*(full_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_I', 'I_0','S_0', 'N_0', 'd_S', 'K_H'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_I = K_I, d_I = d_I, b_I = b_I,
                                   infection_model = "cancer" if b_I >= b_C else "acute")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/S_0
    data.loc[:,"peff_infection"] = data['harm_pI'].to_numpy()/S_0
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/S_0
    data.loc[:,"peff_total_harm"] = data['peff_infection'] + data['peff_toxicity']
    data.loc[:,"peff_scaled_total_harm"] = data['peff_total_harm']/data['harm_pI_noprotection']

    mean_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean())
    std_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).std())
    full_infection_scenarios.append(data)

# stack datasets
pd.concat(mean_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(std_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(full_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/processed_full_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(mean_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(std_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/list_processed_full_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(full_infection_scenarios, f)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 81/81 [57:28<00:00, 42.58s/it]


In [8]:
# Clear memory
del full_df, mean_of_infection_scenarios, std_of_infection_scenarios, full_infection_scenarios